# NSL-KDD Network Intrusion Detection — Binary Classification

**Task:** Normal (0) vs Anomaly/Attack (1)  
**Database:** `nsl_kdd`  
**Tables:** `nsl_kdd_common_train_clean` · `nsl_kdd_common_test_clean`  
**Models:** Logistic Regression · Naive Bayes · KNN · Decision Tree · Random Forest · SVM

---
## 1. Environment Check

In [ ]:
!python --version

---
## 2. Install Dependencies

In [ ]:
%conda install -c anaconda mysql-connector-python -y

In [ ]:
%conda install -c conda-forge imbalanced-learn -y

In [ ]:
# Pin compatible versions of scikit-learn + imbalanced-learn
!pip install scikit-learn==1.4.2 imbalanced-learn==0.12.3 --quiet

---
## 3. MySQL Connection

In [ ]:
import mysql.connector as sql

conn = sql.connect(
    host     = 'localhost',
    user     = 'root',
    password = 'bob123',
    database = 'nsl_kdd',
    use_pure = True
)
conn

In [ ]:
cursor = conn.cursor(buffered=True)
cursor.execute('USE nsl_kdd')
cursor.execute('SHOW TABLES')
for x in cursor:
    print(x)

---
## 4. Load Data from MySQL

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fetch training table
query_train = 'SELECT * FROM nsl_kdd_common_train_clean'
train_raw = pd.read_sql(query_train, conn)

# Fetch test table
query_test = 'SELECT * FROM nsl_kdd_common_test_clean'
test_raw = pd.read_sql(query_test, conn)

print(f'Train : {train_raw.shape[0]:,} rows  x  {train_raw.shape[1]} columns')
print(f'Test  : {test_raw.shape[0]:,} rows  x  {test_raw.shape[1]} columns')
train_raw.head()

In [ ]:
# Close the MySQL connection once data is loaded
conn.close()
print('MySQL connection closed ✓')

---
## 5. Data Overview

In [ ]:
print('=== COLUMN TYPES ===')
print(train_raw.dtypes)
print('\n=== MISSING VALUES ===')
missing = train_raw.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'None ✓')

In [ ]:
train_raw.describe()

---
## 6. Exploratory Data Analysis

In [ ]:
sns.set_theme(style='whitegrid', palette='muted')

# 6.1 Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, title in zip(axes, [train_raw, test_raw], ['Train', 'Test']):
    counts = df['class'].value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=['#4C72B0', '#DD8452'], width=0.5)
    ax.set_title(f'{title} Set — Class Distribution', fontsize=12)
    ax.set_ylabel('Count')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 80,
                f'{v}\n({v / len(df) * 100:.1f}%)',
                ha='center', fontsize=10)
plt.suptitle('Class Distribution: Normal vs Anomaly', fontsize=13)
plt.tight_layout()
plt.show()

ratio = (train_raw['label_bin'].value_counts()[0]
       / train_raw['label_bin'].value_counts()[1])
print(f'Class imbalance ratio (train): {ratio:.1f}:1  ->  SMOTE will be applied')

In [ ]:
# 6.2 Protocol distribution + anomaly rate by protocol
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

train_raw['protocol'].value_counts().plot.bar(
    ax=axes[0], color='steelblue')
axes[0].set_title('Protocol Distribution')
axes[0].set_xlabel('')

ct = (pd.crosstab(train_raw['protocol'], train_raw['class'],
                  normalize='index') * 100)
ct.plot.bar(ax=axes[1], stacked=True,
            color=['#4C72B0', '#DD8452'])
axes[1].set_title('Anomaly Rate by Protocol (%)')
axes[1].set_ylabel('Percentage')
axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# 6.3 Feature distributions by class (log scale)
num_features = ['flow_duration', 'fwd_bytes', 'bwd_bytes',
                'byte_rate', 'packet_rate', 'avg_packet_size']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, num_features):
    for label, grp in train_raw.groupby('class'):
        ax.hist(np.log1p(grp[col]), bins=40,
                alpha=0.5, label=label, density=True)
    ax.set_title(f'log1p({col})')
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions: Normal vs Anomaly (log scale)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 6.4 Pearson correlation heatmap
num_cols = (train_raw
            .select_dtypes(include=np.number)
            .drop(columns=['label_bin'])
            .columns)
corr = train_raw[num_cols].corr()

plt.figure(figsize=(9, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Pearson Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# 6.5 Feature means: Normal vs Anomaly
summary = train_raw.groupby('class')[num_features].mean().T
summary['ratio (anomaly/normal)'] = (
    summary['anomaly'] / summary['normal'].replace(0, np.nan)
).round(1)
print('Feature means by class:')
print(summary.to_string())

---
## 7. Preprocessing Pipeline

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

# 7.1 Drop zero-variance features
# These 5 are constant 0 in NSL-KDD — per-packet TCP flag
# counts are not captured by this dataset format
ZERO_VAR = [
    'fwd_packet_count', 'bwd_packet_count',
    'syn_flag_count',   'rst_flag_count',   'ack_flag_count'
]
train = train_raw.drop(columns=ZERO_VAR).copy()
test  = test_raw.drop(columns=ZERO_VAR).copy()

print(f'Dropped zero-variance: {ZERO_VAR}')
print(f'Remaining columns    : {train.shape[1]}')

In [ ]:
# 7.2 Label-encode categorical features
# Fit on combined train + test vocabulary so that 10 unseen
# service_state values in test do not raise errors at inference
le_proto = LabelEncoder()
le_svc   = LabelEncoder()

le_proto.fit(
    pd.concat([train['protocol'], test['protocol']]).unique())
le_svc.fit(
    pd.concat([train['service_state'], test['service_state']]).unique())

for df in [train, test]:
    df['protocol']      = le_proto.transform(df['protocol'])
    df['service_state'] = le_svc.transform(df['service_state'])

print(f'protocol classes    : {le_proto.classes_}')
print(f'service_state vocab : {len(le_svc.classes_)} unique values')

In [ ]:
# 7.3 Log1p transformation
# Reduces right skew — anomaly traffic has extreme byte values
# e.g. fwd_bytes up to 693 million in flood attacks
LOG_COLS = [
    'flow_duration', 'fwd_bytes',   'bwd_bytes',
    'packet_rate',   'byte_rate',   'avg_packet_size'
]
for col in LOG_COLS:
    train[f'log_{col}'] = np.log1p(train[col])
    test[f'log_{col}']  = np.log1p(test[col])

print('log1p transformation applied ✓')

In [ ]:
# 7.4 Feature matrix and target
FEATURE_COLS = [f'log_{c}' for c in LOG_COLS] + ['protocol', 'service_state']

X_train = train[FEATURE_COLS]
y_train = train['label_bin']
X_test  = test[FEATURE_COLS]
y_test  = test['label_bin']

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'X_train : {X_train.shape}  |  X_test : {X_test.shape}')

In [ ]:
# 7.5 StandardScaler — fit on train only, apply to both
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print('StandardScaler applied ✓')

In [ ]:
# 7.6 SMOTE — balance 9.4:1 class imbalance
# Applied to scaled training set ONLY — never to test data
RANDOM_STATE = 42

print(f'Before SMOTE : {dict(pd.Series(y_train).value_counts())}')

sm = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_s, y_train)

print(f'After  SMOTE : {dict(pd.Series(y_res).value_counts())}')

---
## 8. Model Training & 5-Fold Cross-Validation

In [ ]:
from sklearn.linear_model    import LogisticRegression
from sklearn.naive_bayes     import GaussianNB
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 8.1 Define all six classifiers
models = {
    'Logistic Regression' : LogisticRegression(
                                max_iter=500,
                                random_state=RANDOM_STATE),
    'Naive Bayes'         : GaussianNB(),
    'KNN'                 : KNeighborsClassifier(n_neighbors=5),
    'Decision Tree'       : DecisionTreeClassifier(
                                random_state=RANDOM_STATE),
    'Random Forest'       : RandomForestClassifier(
                                n_estimators=100,
                                random_state=RANDOM_STATE,
                                n_jobs=-1),
    'SVM'                 : SVC(
                                probability=True,
                                random_state=RANDOM_STATE),
}

In [ ]:
# 8.2 Stratified 5-fold CV on SMOTE training set
cv = StratifiedKFold(n_splits=5, shuffle=True,
                     random_state=RANDOM_STATE)
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(
        model, X_res, y_res,
        cv=cv, scoring='f1', n_jobs=-1
    )
    cv_results[name] = scores
    print(f'{name:22s}  CV F1 = {scores.mean():.4f} +/- {scores.std():.4f}')

In [ ]:
# 8.3 CV scores bar chart
fig, ax = plt.subplots(figsize=(11, 4))
names  = list(cv_results.keys())
means  = [v.mean() for v in cv_results.values()]
stds   = [v.std()  for v in cv_results.values()]
colors = ['#4C72B0','#55A868','#C44E52',
          '#8172B2','#CCB974','#64B5CD']

bars = ax.bar(names, means, yerr=stds,
              capsize=5, color=colors, alpha=0.85)
ax.set_ylim(0.75, 1.03)
ax.set_ylabel('F1-Score')
ax.set_title('5-Fold CV F1-Scores (SMOTE training set)')
ax.set_xticklabels(names, rotation=15, ha='right')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, m + 0.003,
            f'{m:.4f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 8.4 Train each model on the full SMOTE training set
trained = {}
for name, model in models.items():
    model.fit(X_res, y_res)
    trained[name] = model
    print(f'{name:22s} trained ✓')

---
## 9. Evaluation on Hold-Out Test Set

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve,
    accuracy_score, f1_score, precision_score, recall_score
)

# 9.1 Collect all metrics
metrics_rows = []
all_preds    = {}
all_probas   = {}

for name, model in trained.items():
    preds = model.predict(X_test_s)
    proba = model.predict_proba(X_test_s)[:, 1]
    all_preds[name]  = preds
    all_probas[name] = proba
    metrics_rows.append({
        'Model'    : name,
        'Accuracy' : accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall'   : recall_score(y_test, preds),
        'F1-Score' : f1_score(y_test, preds),
        'AUC-ROC'  : roc_auc_score(y_test, proba),
    })

metrics_df = pd.DataFrame(metrics_rows).set_index('Model')
print(metrics_df.to_string(float_format='{:.4f}'.format))

In [ ]:
# 9.2 Performance bar chart
metrics_df.plot.bar(
    figsize=(13, 5), ylim=(0.5, 1.05),
    colormap='tab10', edgecolor='white', linewidth=0.4
)
plt.title('Test Set Performance Metrics — All Models')
plt.ylabel('Score')
plt.xticks(rotation=15, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 9.3 Confusion matrices — all six models
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, preds) in zip(axes.flat, all_preds.items()):
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(
        cm, display_labels=['Normal', 'Anomaly']
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=11)
plt.suptitle('Confusion Matrices — All Models (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 9.4 ROC curves
plt.figure(figsize=(8, 6))
colors = ['#4C72B0','#55A868','#C44E52',
          '#8172B2','#CCB974','#64B5CD']
for (name, proba), color in zip(all_probas.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr,
             label=f'{name} (AUC = {auc:.4f})',
             color=color, lw=2)
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 9.5 Full classification reports
for name, preds in all_preds.items():
    print(f'\n=== {name} ===')
    print(classification_report(
        y_test, preds,
        target_names=['Normal', 'Anomaly']
    ))

---
## 10. Without SMOTE — Comparison

In [ ]:
# 10.1 Train same six models without SMOTE
no_smote_rows = []

models_ns = {
    'Logistic Regression' : LogisticRegression(
                                max_iter=500,
                                random_state=RANDOM_STATE),
    'Naive Bayes'         : GaussianNB(),
    'KNN'                 : KNeighborsClassifier(n_neighbors=5),
    'Decision Tree'       : DecisionTreeClassifier(
                                random_state=RANDOM_STATE),
    'Random Forest'       : RandomForestClassifier(
                                n_estimators=100,
                                random_state=RANDOM_STATE,
                                n_jobs=-1),
    'SVM'                 : SVC(
                                probability=True,
                                random_state=RANDOM_STATE),
}

for name, model in models_ns.items():
    model.fit(X_train_s, y_train)
    p  = model.predict(X_test_s)
    pr = model.predict_proba(X_test_s)[:, 1]
    no_smote_rows.append({
        'Model'    : name,
        'Accuracy' : accuracy_score(y_test, p),
        'Precision': precision_score(y_test, p),
        'Recall'   : recall_score(y_test, p),
        'F1-Score' : f1_score(y_test, p),
        'AUC-ROC'  : roc_auc_score(y_test, pr),
    })

no_smote_df = pd.DataFrame(no_smote_rows).set_index('Model')
print('Without SMOTE:')
print(no_smote_df.to_string(float_format='{:.4f}'.format))

In [ ]:
# 10.2 Side-by-side F1 comparison
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(models))
w = 0.35
ax.bar(x - w / 2, no_smote_df['F1-Score'], w,
       label='Without SMOTE', color='#4C72B0', alpha=0.8)
ax.bar(x + w / 2, metrics_df['F1-Score'],  w,
       label='With SMOTE',    color='#DD8452', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(list(models.keys()), rotation=15, ha='right')
ax.set_ylabel('F1-Score (Anomaly Class)')
ax.set_title('F1-Score Comparison: Without vs With SMOTE')
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

---
## 11. Feature Importance (Random Forest)

In [ ]:
rf_model = trained['Random Forest']
fi = pd.Series(
    rf_model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
bar_colors = plt.cm.RdYlGn(fi.values / fi.values.max())
bars = plt.barh(fi.index, fi.values, color=bar_colors)
plt.xlabel('Feature Importance (Gini)')
plt.title('Random Forest — Feature Importances')
for bar, val in zip(bars, fi.values):
    plt.text(val + 0.002,
             bar.get_y() + bar.get_height() / 2,
             f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Top 5 most important features:')
for feat, imp in fi.sort_values(ascending=False).head(5).items():
    print(f'  {feat:28s}: {imp:.4f}')

---
## 12. Final Summary

In [ ]:
print('=' * 65)
print('  FINAL RESULTS — With SMOTE  |  Test set n = 2,803')
print('=' * 65)
print(metrics_df.to_string(float_format='{:.4f}'.format))
print('=' * 65)

best = metrics_df['F1-Score'].idxmax()
row  = metrics_df.loc[best]
print(f'\n  Best model  : {best}')
print(f'  Accuracy    : {row["Accuracy"]:.4f}')
print(f'  Precision   : {row["Precision"]:.4f}')
print(f'  Recall      : {row["Recall"]:.4f}')
print(f'  F1-Score    : {row["F1-Score"]:.4f}')
print(f'  AUC-ROC     : {row["AUC-ROC"]:.4f}')
print()
print('  Pipeline summary')
print('  Data source : MySQL  nsl_kdd -> nsl_kdd_common_train/test_clean')
print('  Dropped     : 5 zero-variance features')
print('  Encoding    : LabelEncoder on protocol + service_state')
print('  Transform   : log1p on 6 continuous features')
print('  Scaling     : StandardScaler (fit on train only)')
print('  Resampling  : SMOTE  9.4:1  ->  1:1  (18,552 training samples)')